In [1]:
using Random;

Random.seed!(123456);

using PanGraph;
using JSON;
using LinearAlgebra;

include("src/interval.jl");
include("src/counter.jl");
include("src/node.jl");
include("src/util.jl");
include("src/block.jl");
include("src/path.jl");
include("src/edge.jl");
include("src/junction.jl");
include("src/cmd.jl");
include("src/mash.jl");
include("src/align.jl");
include("src/mash.jl");

using ..Mash;

In [2]:
Q = function(D)
    n = size(D,1)
    q = (n-2)*D .- sum(D,dims=1) .- sum(D,dims=2)
    q[diagind(q)] .= Inf
    return q
end

#1 (generic function with 1 method)

In [3]:
D = [
  0.0 4.0 7.0 8.0
  4.0 0.0 6.0 5.0
  7.0 6.0 0.0 3.0
  8.0 5.0 3.0 0.0
]

4×4 Matrix{Float64}:
 0.0  4.0  7.0  8.0
 4.0  0.0  6.0  5.0
 7.0  6.0  0.0  3.0
 8.0  5.0  3.0  0.0

In [4]:
sum(D,dims=1) .- sum(D,dims=2)

4×4 Matrix{Float64}:
 0.0  -4.0  -3.0  -3.0
 4.0   0.0   1.0   1.0
 3.0  -1.0   0.0   0.0
 3.0  -1.0   0.0   0.0

In [5]:
q = Q(D)

4×4 Matrix{Float64}:
  Inf   -26.0  -21.0  -19.0
 -26.0   Inf   -19.0  -21.0
 -21.0  -19.0   Inf   -26.0
 -19.0  -21.0  -26.0   Inf

In [6]:
pair = function(Q)
    ι  = argmin(Q)
    q₀ = Q[ι] 
    if ι.I[1] > ι.I[2]
        return ι.I[2], ι.I[1], q₀
    end
    return ι.I[1], ι.I[2], q₀
end

#3 (generic function with 1 method)

In [7]:
pair(q)

(1, 2, -26.0)

In [8]:
dist = function(D, i, j)
    n  = size(D,1)
    d₁ = .5*D[i,j] + 1/(2*(n-2))*(sum(D[i,:]) - sum(D[j,:])) 
    d₂ = D[i,j] - d₁

    if d₁ < 0
        d₂ -= d₁
        d₁ = 0
    end

    if d₂ < 0
        d₁ -= d₂
        d₂ = 0
    end

    dₙ = .5*(D[i,:] .+ D[j,:] .+ D[i,j])

    return d₁, d₂, dₙ
end
dist(D, 2, 3)

(2.75, 3.25, [8.5, 6.0, 6.0, 7.0])

In [9]:
i = 2
j = 3
n = 4
dₙ = .5*(D[i,:] .+ D[j,:] .+ D[i,j])
dₙ

4-element Vector{Float64}:
 8.5
 6.0
 6.0
 7.0

In [10]:
D[i,j]

6.0

In [11]:
D[i,:]

4-element Vector{Float64}:
 4.0
 0.0
 6.0
 5.0

In [12]:
D[j,:]

4-element Vector{Float64}:
 7.0
 6.0
 0.0
 3.0

In [13]:
D[i,:] .+ D[j,:]

4-element Vector{Float64}:
 11.0
  6.0
  6.0
  8.0

In [14]:
D[i,:] .+ D[j,:] .+ D[i, j]

4-element Vector{Float64}:
 17.0
 12.0
 12.0
 14.0

In [15]:
mutable struct Clade
    name   :: String
    parent :: Union{Clade,Nothing}
    left   :: Union{Clade,Nothing}
    right  :: Union{Clade,Nothing}
end

Clade(name) = Clade(name, nothing, nothing, nothing)

function Clade(left::Clade, right::Clade)
    parent = Clade("", nothing, left, right)

    left.parent = parent
    right.parent = parent

    parent
end

names = ["A", "B", "C", "D"]
nodes = [Clade(names[i]) for i in 1:size(D,1)]

join! = function(D)
    q = Q(D)
    i, j, q₀ = pair(q)

    node = Clade(nodes[i], nodes[j])
    nodes[i].parent = node
    nodes[j].parent = node

    d₁, d₂, dₙ = dist(D, i, j)
    D[i,:] .= dₙ
    D[:,i] .= dₙ
    D[i,i]  = 0

    nodes[i] = node

    D = D[1:end .!= j, 1:end .!= j]
    deleteat!(nodes, j)

    return D
end

#9 (generic function with 1 method)

In [16]:
D

4×4 Matrix{Float64}:
 0.0  4.0  7.0  8.0
 4.0  0.0  6.0  5.0
 7.0  6.0  0.0  3.0
 8.0  5.0  3.0  0.0

In [17]:
D[1:end .!= j, 1:end .!= j]

3×3 Matrix{Float64}:
 0.0  4.0  8.0
 4.0  0.0  5.0
 8.0  5.0  0.0

In [18]:
D

4×4 Matrix{Float64}:
 0.0  4.0  7.0  8.0
 4.0  0.0  6.0  5.0
 7.0  6.0  0.0  3.0
 8.0  5.0  3.0  0.0

In [19]:
JSON.print(nodes, 2)

[
  {
    "name": "A",
    "parent": null,
    "left": null,
    "right": null
  },
  {
    "name": "B",
    "parent": null,
    "left": null,
    "right": null
  },
  {
    "name": "C",
    "parent": null,
    "left": null,
    "right": null
  },
  {
    "name": "D",
    "parent": null,
    "left": null,
    "right": null
  }
]


In [20]:
D = join!(D)

3×3 Matrix{Float64}:
 0.0  8.5  8.5
 8.5  0.0  3.0
 8.5  3.0  0.0

In [21]:
JSON.print(nodes, 2)

[
  {
    "name": "",
    "parent": null,
    "left": {
      "name": "A",
      "parent": null,
      "left": null,
      "right": null
    },
    "right": {
      "name": "B",
      "parent": null,
      "left": null,
      "right": null
    }
  },
  {
    "name": "C",
    "parent": null,
    "left": null,
    "right": null
  },
  {
    "name": "D",
    "parent": null,
    "left": null,
    "right": null
  }
]


In [22]:
D = join!(D)

2×2 Matrix{Float64}:
  0.0  10.0
 10.0   0.0

In [23]:
JSON.print(nodes, 2)

[
  {
    "name": "",
    "parent": null,
    "left": {
      "name": "",
      "parent": null,
      "left": {
        "name": "A",
        "parent": null,
        "left": null,
        "right": null
      },
      "right": {
        "name": "B",
        "parent": null,
        "left": null,
        "right": null
      }
    },
    "right": {
      "name": "C",
      "parent": null,
      "left": null,
      "right": null
    }
  },
  {
    "name": "D",
    "parent": null,
    "left": null,
    "right": null
  }
]
